### Tumbling Window

In [ ]:
import os
from pyflink.table import EnvironmentSettings, TableEnvironment
import os

from pathlib import Path

from pyflink.java_gateway import get_gateway
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, ExplainDetail

env_settings = EnvironmentSettings.in_streaming_mode()
t_env = StreamTableEnvironment.create(environment_settings=env_settings)
jar_path = Path("../nexmark/flink-sql-connector-kafka-4.0.0-2.0.jar").resolve().as_uri()

t_env.get_config().set("pipeline.jars", jar_path)

source_ddl = """
CREATE TABLE event_source (
    `key` INT,
    `eventTime` BIGINT,
    `sequenceNumber` BIGINT,
    `payload` STRING,
    `bid` BIGINT,
    `event_timestamp` AS TO_TIMESTAMP_LTZ(`eventTime`, 3),
    WATERMARK FOR `event_timestamp` AS `event_timestamp` - INTERVAL '5' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'event-demo',
    'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
    'properties.group.id' = 'tumbling-window',
    'scan.startup.mode' = 'earliest-offset',
    'sink.partitioner' = 'round-robin',
    'format' = 'json'
);
"""
t_env.execute_sql(source_ddl)

sink_ddl = """
CREATE TABLE print_sink (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    event_key INT,
    event_count BIGINT
) WITH (
    'connector' = 'print'
);
"""
t_env.execute_sql(sink_ddl)

query_tumbling_window = """
INSERT INTO print_sink
SELECT
    TUMBLE_START(event_timestamp, INTERVAL '10' SECOND) as window_start,
    TUMBLE_END(event_timestamp, INTERVAL '10' SECOND) as window_end,
    `key` as event_key,
    COUNT(*) as event_count
FROM
    event_source
GROUP BY
    `key`,
    TUMBLE(event_timestamp, INTERVAL '10' SECOND)
"""

t_env.execute_sql(query_tumbling_window)

### Sliding Window

In [ ]:
import os
from pyflink.table import EnvironmentSettings, TableEnvironment
import os

from pathlib import Path

from pyflink.java_gateway import get_gateway
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, ExplainDetail

env_settings = EnvironmentSettings.in_streaming_mode()
t_env = StreamTableEnvironment.create(environment_settings=env_settings)
jar_path = Path("../nexmark/flink-sql-connector-kafka-4.0.0-2.0.jar").resolve().as_uri()

source_ddl = """
CREATE TABLE event_source (
    `key` INT,
    `eventTime` BIGINT,
    `sequenceNumber` BIGINT,
    `payload` STRING,
    `bid` BIGINT,
    `event_timestamp` AS TO_TIMESTAMP_LTZ(`eventTime`, 3),
    WATERMARK FOR `event_timestamp` AS `event_timestamp` - INTERVAL '5' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'event-demo',
    'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
    'properties.group.id' = 'nexmark',
    'scan.startup.mode' = 'earliest-offset',
    'sink.partitioner' = 'round-robin',
    'format' = 'json'
);
"""
t_env.execute_sql(source_ddl)

sink_ddl = """
CREATE TABLE print_sink (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    event_key INT,
    avg_bid DOUBLE
) WITH (
    'connector' = 'print'
);
"""
t_env.execute_sql(sink_ddl)

query_sliding_window = """
INSERT INTO print_sink
SELECT
    HOP_START(event_timestamp, INTERVAL '5' SECOND, INTERVAL '10' SECOND) as window_start,
    HOP_END(event_timestamp, INTERVAL '5' SECOND, INTERVAL '10' SECOND) as window_end,
    `key` as event_key,
    AVG(bid) as avg_bid
FROM
    event_source
GROUP BY
    `key`,
    HOP(event_timestamp, INTERVAL '5' SECOND, INTERVAL '10' SECOND)
"""

t_env.execute_sql(query_sliding_window)